# Aplicación RAG con NVIDIA NIM · Parte A del workshop

**Autor:** Jerónimo Zapata Cuadros (`JCuAdrox`) · SI3003 Inteligencia Artificial, EAFIT

Copia del notebook `02b_rag_app_nvidia_nim_colab.ipynb` del curso, extendida para las tareas **A1–A4**:

| Sección | Tarea | Qué cambia frente al original |
|---|---|---|
| 1–5 | A1 | Se ejecuta con los 4 `DEMO_DOCUMENTS`; ruta del JSON corregida a `data/company_documents.json` |
| 6 | A2 | Corpus propio *Roca Viva* (centro de escalada ficticio, 9 documentos) y elección de `chunk_size` con datos |
| 7 | A3 | System prompt nuevo (cita `[título]`, frase fija, ≤ 3 oraciones) y comparación antes/después |
| 8 | A4 | `build_pipeline` devuelve el vectorstore; `chat()` muestra cada fuente con su score |

**Modelos:** generación `nvidia/nemotron-3.5-lightning-30b-a3b` · embeddings `nvidia/llama-nemotron-embed-vl-1b-v2`.

> La API key se lee de los *Secrets* de Colab o de la variable de entorno `NVIDIA_API_KEY`; nunca se escribe en el notebook.

## 1. Instalar dependencias

En Colab se instalan aquí; en local ya están en el entorno.

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client

## 2. Configurar la API key y los modelos

Además del original, se añaden dos arreglos verificados: `ResilientEmbeddings` (timeout de 12 s, reintentos y caché, porque el endpoint de embeddings a veces devuelve 502 tras ~30 s) y `enable_thinking=False` para Nemotron 3.5.

In [ ]:
import os
import sys
import time
import json
import getpass
import concurrent.futures
from pathlib import Path
from typing import List, Dict

IN_COLAB = "google.colab" in sys.modules


def find_repo_root() -> Path:
    """Raíz del repo (donde está RESPUESTAS.md); en Colab, el directorio actual."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "RESPUESTAS.md").exists():
            return base
    return Path.cwd()


ROOT = find_repo_root()
EVIDENCIAS = ROOT / "evidencias"
EVIDENCIAS.mkdir(exist_ok=True)

NVIDIA_API_KEY = None
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY and IN_COLAB:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/ y expórtala como variable de entorno.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY  # solo en memoria; nunca se imprime
GENERATION_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
print("✅ Configuración NVIDIA NIM lista | raíz:", ROOT.name)

In [ ]:
import gradio as gr
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

from langchain_core.embeddings import Embeddings
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA


class ResilientEmbeddings(Embeddings):
    """Envuelve NVIDIAEmbeddings con timeout corto, reintentos y caché.

    El endpoint de embeddings a veces responde 502 tras ~30 s. En lugar de esperar,
    cortamos a los `timeout` segundos y reintentamos con backoff. Las consultas y los
    chunks ya vistos se sirven desde caché (útil al reconstruir índices en B2/C4).
    """

    def __init__(self, base, timeout: float = 12.0, retries: int = 6, batch_size: int = 16):
        self.base = base
        self.timeout = timeout
        self.retries = retries
        self.batch_size = batch_size
        self._pool = concurrent.futures.ThreadPoolExecutor(max_workers=8)
        self._query_cache: Dict[str, List[float]] = {}
        self._doc_cache: Dict[str, List[float]] = {}
        self.stats = {"llamadas": 0, "reintentos": 0, "cache_consultas": 0, "cache_chunks": 0}

    def _call(self, fn, arg):
        last_error = None
        for attempt in range(1, self.retries + 1):
            self.stats["llamadas"] += 1
            future = self._pool.submit(fn, arg)
            try:
                return future.result(timeout=self.timeout)
            except Exception as error:  # TimeoutError, 502, etc.
                last_error = error
                self.stats["reintentos"] += 1
                print(f"   ↻ embeddings: intento {attempt}/{self.retries} falló ({type(error).__name__})")
                time.sleep(min(2 ** attempt, 10))
        raise RuntimeError(f"Embeddings fallaron tras {self.retries} intentos") from last_error

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        missing = [t for t in dict.fromkeys(texts) if t not in self._doc_cache]
        self.stats["cache_chunks"] += len(texts) - len(missing)
        for i in range(0, len(missing), self.batch_size):
            batch = missing[i:i + self.batch_size]
            for text, vector in zip(batch, self._call(self.base.embed_documents, batch)):
                self._doc_cache[text] = vector
        return [self._doc_cache[t] for t in texts]

    def embed_query(self, text: str) -> List[float]:
        if text in self._query_cache:
            self.stats["cache_consultas"] += 1
            return self._query_cache[text]
        vector = self._call(self.base.embed_query, text)
        self._query_cache[text] = vector
        return vector


def make_embeddings() -> ResilientEmbeddings:
    return ResilientEmbeddings(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))


# Nemotron 3.5 razona por defecto y ese razonamiento gasta max_completion_tokens;
# con enable_thinking=False la respuesta llega completa en `content`.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}


def make_llm(max_completion_tokens: int = 700, thinking: bool = False) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=max_completion_tokens,
        model_kwargs={} if thinking else NO_THINKING,
    )

### 2.1 Verificación: ¿Nemotron 3.5 devuelve respuestas vacías si razona?

Misma pregunta con un presupuesto pequeño (`max_completion_tokens=120`), con y sin razonamiento.

In [ ]:
# Verificación del problema conocido: ¿el razonamiento deja vacía la respuesta?
probe = "Responde en una frase: ¿qué es un vector de embeddings?"
for thinking in (True, False):
    t0 = time.time()
    try:
        msg = make_llm(max_completion_tokens=120, thinking=thinking).invoke(probe)
        content = (msg.content or "").strip()
        reasoning = msg.additional_kwargs.get("reasoning_content") or ""
        finish = (msg.response_metadata or {}).get("finish_reason")
        print(f"thinking={thinking!s:5} | {time.time() - t0:4.1f}s | finish={finish} | "
              f"len(content)={len(content):3} | len(reasoning)={len(reasoning):3} | {content[:90]!r}")
    except Exception as error:
        print(f"thinking={thinking}: {type(error).__name__}: {str(error)[:200]}")

## 3. Cargar los documentos

El texto original dice `data/company_documents.json`, pero la celda leía `Path("company_documents.json")`. Aquí la ruta se arma desde la raíz del repo: `ROOT / "data" / "company_documents.json"`. Para **A1** se usan los `DEMO_DOCUMENTS`; el JSON propio se carga en la sección 6 (A2).

In [ ]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


DATA_PATH = ROOT / "data" / "company_documents.json"   # ruta corregida
if not DATA_PATH.exists() and IN_COLAB:
    # En Colab se descarga el corpus propio desde el repo de entrega.
    import urllib.request
    DATA_PATH.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/JCuAdrox/Inteligencia_Artificial/main/data/company_documents.json", DATA_PATH)

company_documents = DEMO_DOCUMENTS   # A1 corre con los documentos de demostración
print(f"✅ A1: {len(company_documents)} documentos de demostración | A2: {DATA_PATH.relative_to(ROOT)} existe = {DATA_PATH.exists()}")

## 4. Construir el pipeline RAG

Igual al original (splitter 400/40, `NVIDIAEmbeddings(truncate="END")`, Qdrant `:memory:` con `Distance.COSINE`, retriever `similarity_score_threshold` con k=3 y umbral 0.3, LLM con temperature 0.2 y 700 tokens, LCEL con `RunnableParallel` → `{question, documents, answer}`), pero parametrizado para reutilizarlo en A2–A4:

- `chunk_size`, `chunk_overlap`, `system_prompt` y `collection` son argumentos.
- **A4:** devuelve también el `vectorstore`.

In [ ]:
SYSTEM_PROMPT_ORIGINAL = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Si el contexto no contiene la respuesta, "
    "indica que no tienes esa información. Responde en español y de forma concisa."
)

embeddings = make_embeddings()   # se comparte entre índices para aprovechar la caché
EMBEDDING_DIM = len(embeddings.embed_query("prueba de dimensión"))
print("Dimensión de embeddings:", EMBEDDING_DIM)


def split_documents(documents: List[Dict], chunk_size: int = 400, chunk_overlap: int = 40) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))
    return chunks


def format_docs(items: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
        for item in items
    )


def build_pipeline(documents: List[Dict], chunk_size: int = 400, chunk_overlap: int = 40,
                   system_prompt: str = SYSTEM_PROMPT_ORIGINAL, collection: str = "company_docs_gradio",
                   k: int = 3, score_threshold: float = 0.3):
    llm = make_llm(max_completion_tokens=700)
    chunks = split_documents(documents, chunk_size, chunk_overlap)

    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    vectorstore = QdrantVectorStore(client=qdrant_client, collection_name=collection, embedding=embeddings)
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    setup_and_retrieval = RunnableParallel(question=RunnablePassthrough(), documents=retriever)
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks, vectorstore


print("Construyendo el índice A1...")
t0 = time.time()
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo en {time.time() - t0:.1f}s: {len(titles)} documentos, {len(indexed_chunks)} chunks")

## 5. Interfaz Gradio (A1)

`make_chat` reproduce el `chat()` original (fuentes únicas por título). Con `vectorstore` añade la lista de A4 con scores.

In [ ]:
def make_chat(pipeline, vectorstore=None, k_scores: int = 3):
    def chat(question: str, history: list | None):
        history = history or []
        question = (question or "").strip()
        if not question:
            return "", history, "Escribe una pregunta."
        try:
            result = pipeline.invoke(question)
            answer = result["answer"].strip() or "(respuesta vacía)"
            if vectorstore is None:
                unique_sources = []
                for document in result["documents"]:
                    title = document.metadata.get("title", "Sin título")
                    if title not in unique_sources:
                        unique_sources.append(title)
                sources_md = "**Fuentes recuperadas:**\n" + "\n".join(
                    f"- {title}" for title in unique_sources
                ) if unique_sources else "No se encontraron documentos relevantes."
            else:
                scored = vectorstore.similarity_search_with_score(question, k=k_scores)
                sources_md = f"**Fuentes (similarity_search_with_score, k={k_scores}) · score = coseno:**\n" + "\n".join(
                    f"- **{doc.metadata.get('title')}** · chunk {doc.metadata.get('chunk_index')} · score `{score:.3f}`"
                    for doc, score in scored
                )
        except Exception as error:
            answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
            sources_md = f"**Error:** `{type(error).__name__}: {error}`"
        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        return "", history, sources_md
    return chat


def build_ui(chat_fn, titles: List[str], n_chunks: int, examples: List[str],
             header: str = "# RAG empresarial", retrieval: str = "k=3, threshold=0.3"):
    with gr.Blocks(title="RAG con NVIDIA NIM") as demo:
        gr.Markdown(header + "\nConsulta documentos mediante **LangChain + NVIDIA NIM + Qdrant**.")
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### Documentos indexados")
                gr.Markdown("\n".join(f"- {title}" for title in titles))
                gr.Markdown(f"*{n_chunks} chunks en el índice*")
                gr.Markdown("---")
                gr.Markdown(
                    f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                    f"**LLM:** `{GENERATION_MODEL}`  \n"
                    "**Vector store:** Qdrant en memoria  \n"
                    f"**Retrieval:** {retrieval}"
                )
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(height=430, label="Chat")
                with gr.Row():
                    question_box = gr.Textbox(placeholder="Haz una pregunta sobre los documentos...",
                                              label="Pregunta", lines=2, scale=4)
                    send_button = gr.Button("Enviar", variant="primary", scale=1)
                sources_box = gr.Markdown()
                clear_button = gr.Button("Limpiar", variant="secondary")
        gr.Examples(examples=[[e] for e in examples], inputs=question_box, label="Preguntas de ejemplo")
        send_button.click(chat_fn, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
        question_box.submit(chat_fn, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
        clear_button.click(lambda: ([], ""), outputs=[chatbot, sources_box])
    return demo

import subprocess
from IPython.display import Image, display

TOMAR_CAPTURAS = os.getenv("TOMAR_CAPTURAS") == "1"


def launch(demo, port: int):
    """Lanza Gradio en localhost sin bloquear la celda (share=True solo en Colab)."""
    _, local_url, share_url = demo.queue().launch(
        server_name="127.0.0.1",
        server_port=port,
        share=IN_COLAB,
        prevent_thread_lock=True,
        show_error=True,
        quiet=True,
    )
    print("App en", share_url or local_url)
    return local_url


def capture(url: str, filename: str, questions: List[str], dropdown: str | None = None):
    """Captura la app con Playwright (scripts/capture_gradio.py) y la muestra aquí."""
    out = EVIDENCIAS / filename
    if not TOMAR_CAPTURAS:
        print("TOMAR_CAPTURAS != 1: sin captura automática (toma una manual en Colab).")
        return
    cmd = [sys.executable, str(ROOT / "scripts" / "capture_gradio.py"), url, str(out)]
    for question in questions:
        cmd += ["--question", question]
    if dropdown:
        cmd += ["--dropdown", dropdown]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    print(result.stdout.strip() or result.stderr.strip()[-800:])
    if out.exists():
        display(Image(filename=str(out), width=900))

### A1 · Las 4 preguntas de ejemplo y una fuera de contexto

In [ ]:
A1_QUESTIONS = [
    ("¿Cuál es el precio del plan Enterprise de CloudSync Pro?", "$49/mes por usuario con compromiso anual"),
    ("¿Cuántos días pueden trabajar remotamente los empleados?", "Hasta 3 por semana, con aprobación del jefe"),
    ("¿Qué pasa con los reembolsos mayores a $100?", "Aprobación del supervisor; 3 a 5 días hábiles"),
    ("¿Qué ocurre durante la primera semana de onboarding?", "Módulos de seguridad, cumplimiento y cultura"),
    ("¿Quién ganó el Mundial de fútbol de 2022?", "Fuera de contexto: debería decir que no tiene la información"),
]


def short(text: str, n: int = 220) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + "…"


a1_rows = []
for question, expected in A1_QUESTIONS:
    t0 = time.time()
    result = rag_pipeline.invoke(question)
    titles_found = list(dict.fromkeys(d.metadata["title"] for d in result["documents"]))
    a1_rows.append((question, expected, result["answer"], titles_found, time.time() - t0))

print("| # | Pregunta | Esperada | Respuesta del RAG | Fuentes | s |")
print("|---|---|---|---|---|---|")
for i, (q, e, a, t, s) in enumerate(a1_rows, 1):
    print(f"| {i} | {q} | {e} | {short(a)} | {'; '.join(t) or '—'} | {s:.1f} |")

In [ ]:
demo_a1 = build_ui(make_chat(rag_pipeline), titles, len(indexed_chunks),
                   [q for q, _ in A1_QUESTIONS[:4]])
url_a1 = launch(demo_a1, 7861)
capture(url_a1, "A1_gradio.png", [A1_QUESTIONS[0][0], A1_QUESTIONS[4][0]])

## 6. A2 · Corpus propio: *Roca Viva*, centro de escalada ficticio

`data/company_documents.json` tiene 9 documentos (`id`, `title`, `content`) con metadata de **dos claves**: `area` (ventas, seguridad, operaciones, formacion, comunidad) y `tipo`. Cada documento mezcla 5–7 datos concretos (precios, edades, horarios), así que el tamaño del chunk decide si un dato queda junto a su contexto.

**Elección de `chunk_size` con datos:** primero se mide cuántos chunks produce cada tamaño y después qué tamaño pone en el top-1 el chunk con la palabra clave de cada pregunta respondible.

In [ ]:
own_documents = load_documents(DATA_PATH)
lengths = [len(d["content"]) for d in own_documents]
print(f"{len(own_documents)} documentos | caracteres por documento: min={min(lengths)} media={sum(lengths)/len(lengths):.0f} max={max(lengths)}")
print("Claves de metadata:", sorted({k for d in own_documents for k in d["metadata"]}))

A2_QUESTIONS = [
    # (pregunta, respuesta esperada, palabra clave para el chequeo automático, ¿respondible?)
    ("¿Cuánto cuesta el plan Pared Libre?", "$189.000 COP al mes", "189.000", True),
    ("¿Un adolescente de 15 años puede escalar solo?", "Sí, solo en boulder y con consentimiento firmado por un acudiente", "consentimiento", True),
    ("¿Qué evalúa el examen de la certificación de aseguramiento?", "Nudo ocho doble, chequeo cruzado y aseguramiento con GriGri", "grigri", True),
    ("¿A qué hora cierra Roca Viva los domingos?", "5:00 p. m. (último ingreso 45 min antes)", "5:00", True),
    ("¿Por cuánto tiempo puedo congelar mi membresía mensual?", "Entre 15 y 60 días, una vez al año (90 días con incapacidad médica)", "60", True),
    ("¿Roca Viva tiene piscina o sauna?", "No está en los documentos", None, False),
    ("¿Cuánto gana al mes un instructor de Roca Viva?", "No está en los documentos", None, False),
]

for size in (200, 300, 400, 600, 900):
    chunks = split_documents(own_documents, size, int(size * 0.1))
    sizes = [len(c.page_content) for c in chunks]
    print(f"chunk_size={size:>3}: {len(chunks):>2} chunks | media {sum(sizes)/len(sizes):5.0f} caracteres")

In [ ]:
def top1_keyword_hits(chunk_size: int) -> tuple[int, list]:
    chunks = split_documents(own_documents, chunk_size, int(chunk_size * 0.1))
    client = QdrantClient(":memory:")
    client.create_collection("a2_probe", vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name="a2_probe", embedding=embeddings)
    store.add_documents(chunks)
    hits, detail = 0, []
    for question, _, keyword, answerable in A2_QUESTIONS:
        if not answerable:
            continue
        doc, score = store.similarity_search_with_score(question, k=1)[0]
        ok = keyword.lower() in doc.page_content.lower()
        hits += ok
        detail.append((question[:45], doc.metadata["title"][:40], round(score, 3), ok))
    return hits, detail


probe = {}
for size in (200, 400, 900):
    probe[size] = top1_keyword_hits(size)
    print(f"chunk_size={size}: top-1 contiene la palabra clave en {probe[size][0]}/5")
    for row in probe[size][1]:
        print("    ", row)

# Se elige el tamaño con más aciertos; en empate, el más pequeño (contexto más preciso y barato).
A2_CHUNK_SIZE = sorted(probe, key=lambda s: (-probe[s][0], s))[0]
print("\n➡ chunk_size elegido para A2:", A2_CHUNK_SIZE)

In [ ]:
REFUSAL_MARKERS = ["no tengo", "no encuentro", "no contiene", "no dispongo", "no se menciona",
                   "no hay información", "no cuento con", "no se encuentra", "no aparece", "no está"]


def is_refusal(answer: str) -> bool:
    return any(m in answer.lower() for m in REFUSAL_MARKERS)


rag_a2, chunks_a2, vectorstore_a2 = build_pipeline(
    own_documents, chunk_size=A2_CHUNK_SIZE, chunk_overlap=int(A2_CHUNK_SIZE * 0.1), collection="roca_viva")
print(f"Índice A2: {len(chunks_a2)} chunks\n")

a2_rows = []
for question, expected, keyword, answerable in A2_QUESTIONS:
    result = rag_a2.invoke(question)
    answer = result["answer"]
    auto_ok = (keyword.lower() in answer.lower()) if answerable else is_refusal(answer)
    titles_found = list(dict.fromkeys(d.metadata["title"] for d in result["documents"]))
    a2_rows.append(dict(question=question, expected=expected, answer=answer, answerable=answerable,
                        auto_ok=auto_ok, titles=titles_found))

print("| # | Pregunta | ¿Respondible? | Esperada | Respuesta del RAG | Chequeo automático | Fuentes |")
print("|---|---|---|---|---|---|---|")
for i, r in enumerate(a2_rows, 1):
    print(f"| {i} | {r['question']} | {'sí' if r['answerable'] else 'no'} | {r['expected']} | {short(r['answer'])} | "
          f"{'✅' if r['auto_ok'] else '❌'} | {'; '.join(r['titles']) or '—'} |")

In [ ]:
demo_a1.close()
demo_a2 = build_ui(make_chat(rag_a2), [d["title"] for d in own_documents], len(chunks_a2),
                   [q for q, *_ in A2_QUESTIONS[:5]], header="# Roca Viva · asistente del centro de escalada",
                   retrieval=f"chunk_size={A2_CHUNK_SIZE}, k=3, threshold=0.3")
url_a2 = launch(demo_a2, 7862)
capture(url_a2, "A2_corpus.png", [A2_QUESTIONS[2][0], A2_QUESTIONS[5][0]])

## 7. A3 · Ingeniería del prompt (antes y después)

Tres preguntas de A2: **fácil** (precio del plan), **ambigua** («¿cuánto cuesta el curso?», hay dos cursos y clases particulares) y **fuera de contexto** (piscina). Se mide automáticamente cada instrucción: (a) ¿cita un título real entre corchetes?, (b) ¿usa la frase fija exacta cuando no hay información?, (c) ¿≤ 3 oraciones?

In [ ]:
import re
import difflib

FIXED_PHRASE = "No encuentro esa información en los documentos de Roca Viva."
SYSTEM_PROMPT_A3 = (
    "Eres el asistente de Roca Viva. Responde únicamente con información del CONTEXTO. "
    "Reglas obligatorias: "
    "(a) Cita entre corchetes el título exacto del documento que respalda la respuesta, por ejemplo [Planes de membresía Roca Viva]. "
    f"(b) Si el contexto no alcanza para responder, responde exactamente: \"{FIXED_PHRASE}\" y nada más. "
    "(c) Usa como máximo 3 oraciones. Responde en español."
)

A3_QUESTIONS = [
    ("fácil", "¿Cuánto cuesta el plan Pared Libre?"),
    ("ambigua", "¿Cuánto cuesta el curso?"),
    ("fuera de contexto", "¿Roca Viva tiene piscina o sauna?"),
]
own_titles = {d["title"] for d in own_documents}


def count_sentences(text: str) -> int:
    parts = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¿¡\[])", text.strip())
    return len([p for p in parts if p.strip()])


def check(answer: str) -> dict:
    cited = [t for t in re.findall(r"\[([^\]]+)\]", answer) if t.strip() in own_titles]
    return {"cita": bool(cited), "frase_fija": FIXED_PHRASE.lower() in answer.lower(),
            "oraciones": count_sentences(answer)}


rag_a3, _, _ = build_pipeline(own_documents, chunk_size=A2_CHUNK_SIZE, chunk_overlap=int(A2_CHUNK_SIZE * 0.1),
                              system_prompt=SYSTEM_PROMPT_A3, collection="roca_viva_a3")
a3_rows = []
for kind, question in A3_QUESTIONS:
    before = rag_a2.invoke(question)["answer"]
    after = rag_a3.invoke(question)["answer"]
    a3_rows.append((kind, question, before, check(before), after, check(after)))

print("| Tipo | Pregunta | Antes | ¿cita? / ¿frase? / oraciones | Después | ¿cita? / ¿frase? / oraciones |")
print("|---|---|---|---|---|---|")
for kind, q, b, cb, a, ca in a3_rows:
    fmt = lambda c: f"{'sí' if c['cita'] else 'no'} / {'sí' if c['frase_fija'] else 'no'} / {c['oraciones']}"
    print(f"| {kind} | {q} | {short(b, 260)} | {fmt(cb)} | {short(a, 260)} | {fmt(ca)} |")

In [ ]:
print("Diff del system prompt:\n")
for line in difflib.unified_diff(
        [s.strip() + "." for s in SYSTEM_PROMPT_ORIGINAL.split(". ") if s.strip()],
        [s.strip() for s in re.split(r"(?<=\.) (?=\(|[A-Z])", SYSTEM_PROMPT_A3) if s.strip()],
        fromfile="SYSTEM_PROMPT_ORIGINAL", tofile="SYSTEM_PROMPT_A3", lineterm=""):
    print(line)

## 8. A4 · Score de similitud en las fuentes

`build_pipeline` ya devuelve el `vectorstore` y `chat()` llama a `similarity_search_with_score(question, k=3)`. Primero se comprueba **en qué escala** está cada score, porque el `score_threshold` del retriever no se compara contra el mismo número.

In [ ]:
import numpy as np

question = A2_QUESTIONS[0][0]
with_score = vectorstore_a2.similarity_search_with_score(question, k=3)
with_relevance = vectorstore_a2.similarity_search_with_relevance_scores(question, k=3)
q_vec = np.array(embeddings.embed_query(question))

print("| chunk | coseno numpy | similarity_search_with_score | ..._with_relevance_scores | (coseno+1)/2 |")
print("|---|---|---|---|---|")
for (doc, s), (_, r) in zip(with_score, with_relevance):
    d_vec = np.array(embeddings.embed_documents([doc.page_content])[0])
    cos = float(q_vec @ d_vec / (np.linalg.norm(q_vec) * np.linalg.norm(d_vec)))
    print(f"| {doc.metadata['title'][:35]} #{doc.metadata['chunk_index']} | {cos:.4f} | {s:.4f} | {r:.4f} | {(cos + 1) / 2:.4f} |")

In [ ]:
A4_QUESTIONS = [(q, ans) for q, _, _, ans in [A2_QUESTIONS[i] for i in (0, 2, 3, 5, 6)]]

a4_rows = []
for question, answerable in A4_QUESTIONS:
    scored = vectorstore_a2.similarity_search_with_score(question, k=3)
    a4_rows.append((question, answerable, [(d.metadata["title"], s) for d, s in scored]))

print("| Pregunta | Tipo | score #1 | score #2 | score #3 | Fuente #1 |")
print("|---|---|---|---|---|---|")
for q, ok, sc in a4_rows:
    print(f"| {q} | {'buena' if ok else 'fuera'} | " + " | ".join(f"{s:.3f}" for _, s in sc) + f" | {sc[0][0]} |")

good_top1 = [sc[0][1] for _, ok, sc in a4_rows if ok]
bad_top1 = [sc[0][1] for _, ok, sc in a4_rows if not ok]
good_all = [s for _, ok, sc in a4_rows if ok for _, s in sc]
print(f"\ntop-1 buenas: min={min(good_top1):.3f} | top-1 fuera de contexto: max={max(bad_top1):.3f}")
if min(good_top1) > max(bad_top1):
    PROPOSED_COS = round((min(good_top1) + max(bad_top1)) / 2, 3)
    print(f"Hay separación: umbral propuesto (coseno) = punto medio = {PROPOSED_COS}")
else:
    PROPOSED_COS = round(min(good_top1), 3)
    print(f"No hay separación limpia; el umbral no basta para filtrar. Referencia: {PROPOSED_COS}")
PROPOSED_RELEVANCE = round((PROPOSED_COS + 1) / 2, 3)
print(f"Equivalente en la escala del retriever (relevancia=(cos+1)/2): score_threshold = {PROPOSED_RELEVANCE}")

In [ ]:
# ¿Qué filtra cada umbral en el retriever? (el original usa 0.3 en escala de relevancia)
for label, thr in [("original 0.3", 0.3), (f"propuesto {PROPOSED_RELEVANCE}", PROPOSED_RELEVANCE)]:
    retr = vectorstore_a2.as_retriever(search_type="similarity_score_threshold",
                                        search_kwargs={"k": 3, "score_threshold": thr})
    counts = [len(retr.invoke(q)) for q, _ in A4_QUESTIONS]
    print(f"umbral {label:>16}: chunks devueltos por pregunta = {counts}  (3 buenas, 2 fuera)")

In [ ]:
demo_a2.close()
demo_a4 = build_ui(make_chat(rag_a2, vectorstore=vectorstore_a2), [d["title"] for d in own_documents], len(chunks_a2),
                   [q for q, _ in A4_QUESTIONS], header="# Roca Viva · fuentes con score",
                   retrieval=f"k=3, scores con similarity_search_with_score")
url_a4 = launch(demo_a4, 7863)
capture(url_a4, "A4_scores.png", [A4_QUESTIONS[1][0]])

In [ ]:
print("Estadísticas de ResilientEmbeddings:", embeddings.stats)
demo_a4.close()
gr.close_all()